In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dhivyeshrk/diseases-and-symptoms-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv


# Symptom-to-Disease Classifier — Model Training

## Dataset
This notebook uses the **Diseases and Symptoms Dataset**, one of the largest publicly 
available symptom-disease datasets on Kaggle:
- **246,945 patient records**
- **773 unique diseases**
- **377 binary symptom features** (one-hot encoded: 1 = symptom present, 0 = absent)
- Source: Kaggle

## What this notebook does
1. **Exploratory Data Analysis (EDA)** — inspect class distribution, check for missing/imbalanced data
2. **Preprocessing** — handle class imbalance, train/test split
3. **Model Training** — train and compare classifiers on the symptom data
4. **Evaluation** — accuracy, precision/recall, confusion matrix analysis
5. **Export** — save the trained model, along with any encoders, as reusable artifacts (`.pkl`) for downstream deployment

### STEP 1: LOAD DATASET

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/dhivyeshrk/diseases-and-symptoms-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv


In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/dhivyeshrk/diseases-and-symptoms-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv')

# Basic info
print("Dataset shape:", df.shape)
print("\nNumber of unique diseases:", df['diseases'].nunique())
print("\nFirst 5 rows:")
df.head()

Dataset shape: (246945, 378)

Number of unique diseases: 773

First 5 rows:


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


### Note on `.head()` output
`.head()` only displays the first 5 rows for preview purposes — it does not reflect 
the entire dataset. Since this dataset is grouped by disease, the first several 
rows all belong to the same class (`panic disorder`), which is why `.head()` 
initially showed only one disease name.

Verified using `df.sample(10)` to confirm the full dataset contains all 773 
diseases distributed throughout, not just the first group.

**Confirmed dataset stats:**
- Total rows: 246,945
- Total columns: 378 (1 disease label + 377 binary symptom features)
- Unique diseases: 773
- Missing values: 0

### STEP 2: EXPLORATORY DATA ANALYSIS (EDA)

In [4]:
# Check for missing values
print("Missing values:", df.isnull().sum().sum())

# Class distribution
disease_counts = df['diseases'].value_counts()
print("\nMost common diseases (top 10):")
print(disease_counts.head(10))

print("\nRarest diseases (bottom 10):")
print(disease_counts.tail(10))

print("\nDiseases with only 1 sample:", (disease_counts == 1).sum())
print("Diseases with less than 5 samples:", (disease_counts < 5).sum())

Missing values: 0

Most common diseases (top 10):
diseases
cystitis                          1219
nose disorder                     1218
vulvodynia                        1218
complex regional pain syndrome    1217
spondylosis                       1216
vaginal cyst                      1215
esophagitis                       1215
peripheral nerve disorder         1215
hypoglycemia                      1215
conjunctivitis due to allergy     1215
Name: count, dtype: int64

Rarest diseases (bottom 10):
diseases
diabetes                  1
thalassemia               1
heat stroke               1
gas gangrene              1
typhoid fever             1
open wound of the head    1
myocarditis               1
chronic ulcer             1
hypergammaglobulinemia    1
kaposi sarcoma            1
Name: count, dtype: int64

Diseases with only 1 sample: 19
Diseases with less than 5 samples: 52


### Problem Identified: Class Imbalance

The dataset has severe class imbalance across the 773 disease labels:

- Most common diseases have **~1,200+ samples** each (e.g., cystitis, nose disorder)
- **52 diseases** have fewer than 5 samples
- **19 diseases** have only **1 sample** each (e.g., diabetes, typhoid fever, thalassemia)

**Why this is a problem:**
- A model cannot reliably learn a disease pattern from just 1–4 examples
- Diseases with only 1 sample cause errors during train/test splitting, since a 
  stratified split requires at least 2 samples per class
- Training on such extreme imbalance risks the model ignoring rare classes entirely 
  and overfitting to the frequent ones

**Solution:** Filter out diseases with fewer than a minimum sample threshold, 
keeping only diseases with enough data to train and evaluate reliably.

### STEP 3: PREPROCESSING — HANDLING CLASS IMBALANCE

This step removes diseases with insufficient samples to train and evaluate reliably. 
A minimum threshold of 10 samples per disease is applied — diseases below this 
threshold are excluded, while all diseases with enough data are retained for training.

In [5]:
# Remove diseases with very few samples (can't reliably train/test on them)
MIN_SAMPLES = 10

disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= MIN_SAMPLES].index

print(f"Diseases before filtering: {df['diseases'].nunique()}")
df_filtered = df[df['diseases'].isin(valid_diseases)].copy()
print(f"Diseases after filtering (>= {MIN_SAMPLES} samples): {df_filtered['diseases'].nunique()}")
print(f"Rows before: {len(df)}, Rows after: {len(df_filtered)}")


Diseases before filtering: 773
Diseases after filtering (>= 10 samples): 677
Rows before: 246945, Rows after: 246512


### Result

Filtering removed only 96 out of 773 diseases (773 → 677 remaining), while 
preserving 246,512 out of 246,945 rows — a loss of just 433 rows (0.2% of the data).

This means the dataset had very few diseases with insufficient samples, and the 
vast majority of the original data — along with 677 diseases — is retained for 
training a reliable model.

### STEP 4: TRAIN/TEST SPLIT

In [6]:
from sklearn.model_selection import train_test_split

X = df_filtered.drop('diseases', axis=1)
y = df_filtered['diseases']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Training samples: 197209
Testing samples: 49303
Number of features: 377


### Result

The dataset was split into 197,209 training samples and 49,303 testing samples 
(80/20 split), with 377 symptom features per sample. The `stratify=y` parameter 
ensures each disease is proportionally represented in both the training and 
testing sets, which is especially important given the class imbalance handled 
in the previous step.

### STEP 5: MODEL TRAINING

A Random Forest classifier is used for this task. Random Forest works well with 
binary/tabular features like ours (377 symptom columns), handles non-linear 
relationships between symptoms, and is more robust to class imbalance compared 
to simpler models like Logistic Regression.

In [7]:
from sklearn.ensemble import RandomForestClassifier
import time

start_time = time.time()

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=30,
    random_state=42,
    n_jobs=2,
    class_weight='balanced'
)

model.fit(X_train, y_train)

print(f"Training completed in {time.time() - start_time:.2f} seconds")

Training completed in 14.33 seconds


### Result

Model training completed successfully in ~14 seconds. The initial configuration 
(`n_estimators=100`, unlimited depth) caused a memory overflow due to the large 
number of classes (677) and features (377). Reducing to `n_estimators=50`, 
`max_depth=30`, and limiting parallel jobs (`n_jobs=2`) resolved the issue while 
keeping training fast.

### STEP 6: MODEL EVALUATION

In [8]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions on test set
y_pred = model.predict(X_test)

# Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Detailed report (precision, recall, f1 per class) - just top-level summary
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
print(f"\nMacro Avg F1-score: {report['macro avg']['f1-score']:.4f}")
print(f"Weighted Avg F1-score: {report['weighted avg']['f1-score']:.4f}")

Test Accuracy: 0.6724 (67.24%)

Macro Avg F1-score: 0.7234
Weighted Avg F1-score: 0.7200


### STEP 7: TOP-3 ACCURACY CHECK

Since the model predicts across 677 diseases, checking only the single top prediction 
(67.24% accuracy) may be an overly strict metric. Real-world symptom checkers typically 
show multiple probable conditions rather than a single diagnosis. This step checks 
whether the correct disease appears within the model's top-3 predictions.

In [9]:
import numpy as np

y_proba = model.predict_proba(X_test)
classes = model.classes_

top3_preds = np.argsort(y_proba, axis=1)[:, -3:]

correct_top3 = 0
for i, true_label in enumerate(y_test.values):
    top3_labels = classes[top3_preds[i]]
    if true_label in top3_labels:
        correct_top3 += 1

top3_accuracy = correct_top3 / len(y_test)
print(f"Top-3 Accuracy: {top3_accuracy:.4f} ({top3_accuracy*100:.2f}%)")

Top-3 Accuracy: 0.7668 (76.68%)


### Result

Top-3 accuracy is 76.68%, compared to 67.24% for exact top-1 prediction. This means 
that in roughly 3 out of 4 cases, the correct disease appears among the model's 
top-3 suggestions — even when it isn't the single highest-confidence prediction.

This supports designing the final application around a **differential diagnosis** 
approach (showing top-3 probable conditions with confidence scores) rather than a 
single definitive answer, which is both more clinically realistic and more reliable 
given the large number of disease classes (677).

### STEP 8: TRYING XGBOOST FOR IMPROVED ACCURACY

XGBoost (Gradient Boosting) is tried as an alternative to Random Forest. Unlike 
Random Forest, which builds trees independently, XGBoost builds trees sequentially — 
each new tree corrects errors made by previous ones. This often results in better 
accuracy on structured/tabular data like this symptom dataset.

### Note
XGBoost training was attempted but caused repeated memory allocation issues and 
long processing times on the Kaggle environment, even with reduced parameters. 
Given time constraints, the project proceeds with the Random Forest model 
(67.24% top-1 accuracy, 76.68% top-3 accuracy) as the final model.

### STEP 9: SAVE MODEL FOR DEPLOYMENT

In [10]:
import joblib

joblib.dump(model, '/kaggle/working/disease_prediction_model.pkl')
joblib.dump(list(valid_diseases), '/kaggle/working/disease_labels.pkl')
joblib.dump(list(X_train.columns), '/kaggle/working/symptom_columns.pkl')

print("Model and artifacts saved successfully!")

Model and artifacts saved successfully!
